In [5]:
import pandas as pd
import statsmodels.api as sm
import numpy as np

In [3]:
df = pd.read_excel('data/innovation_new.xlsx')

In [4]:
df

,Sales,growth_technological,growth_marketing,share_RD,share_equipments,patents,cooperation,support,lab,joint,climate,share_technology,emp,growth_innovation,fin_support,age,compet
0,3.59,11.67,7.82,34.100000,36.900000,10,1,0,1,0,6,0.80,0.48,12.86,0.000000,43.000000,0
1,1.28,2.34,4.70,27.100000,16.560000,1,0,0,0,0,10,0.02,0.46,2.29,0.000000,43.000000,1
2,0.48,11.73,7.07,7.900000,31.320000,0,0,0,0,0,5,0.06,0.38,11.79,0.000000,35.000000,837
3,0.86,9.01,3.91,19.000000,16.290000,3,0,0,0,0,5,0.31,0.10,8.95,0.000000,24.000000,1
4,2.24,8.93,4.20,31.200000,32.400000,3,1,1,0,0,8,0.07,0.36,9.94,0.897416,39.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1.09,6.81,3.84,19.100000,43.470000,0,0,0,0,0,5,0.32,0.12,6.79,0.000000,43.000000,1
996,1.48,5.89,4.03,26.400000,18.360000,0,0,0,0,0,9,0.03,0.26,6.90,0.000000,36.000000,0
997,1.03,12.61,3.96,13.400000,28.620000,0,1,0,0,0,4,0.01,0.11,13.56,0.000000,24.000000,1
998,11.71,28.44,3.80,43.627365,14.205606,0,0,0,1,1,10,0.75,0.26,29.09,0.000000,39.311698,0


### 1. Линейная регерессия

In [6]:
y = pd.to_numeric(df['Sales'], errors='coerce')
X = df.drop(columns=['Sales']).apply(pd.to_numeric, errors='coerce')
data = pd.concat([y, X], axis=1).dropna()
y = data['Sales'].values
X = data.drop(columns=['Sales']).values
feature_names = data.drop(columns=['Sales']).columns.tolist()

стандартизуем признаки

In [7]:
X_mean = X.mean(axis=0)
X_std = X.std(axis=0, ddof=1)
Xz = (X - X_mean) / X_std

Ковариационная матрица и собственные значения

In [8]:
cov_matrix = np.cov(Xz, rowvar=False)
eig_vals, eig_vecs = np.linalg.eig(cov_matrix)
idx = np.argsort(eig_vals)[::-1]
eig_vals = eig_vals[idx]
eig_vecs = eig_vecs[:, idx]

Выбор числа главных компонент для 85% суммарной дисперсии

In [9]:
explained_ratio = eig_vals / eig_vals.sum()
cum_ratio = np.cumsum(explained_ratio)
m = int(np.argmax(cum_ratio >= 0.85) + 1)
m


10

In [10]:
PCA_coef = eig_vecs[:, :m]
X_pca = Xz.dot(PCA_coef)


Регрессия Sales на выбранные главные компоненты

In [11]:
X_pca_const = sm.add_constant(X_pca, has_constant='add')
model_pcr = sm.OLS(y, X_pca_const).fit()
summary_text = model_pcr.summary().as_text()
print(summary_text)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.807
Model:                            OLS   Adj. R-squared:                  0.805
Method:                 Least Squares   F-statistic:                     414.3
Date:                Sat, 25 Oct 2025   Prob (F-statistic):               0.00
Time:                        21:27:24   Log-Likelihood:                -1278.7
No. Observations:                1000   AIC:                             2579.
Df Residuals:                     989   BIC:                             2633.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.9135      0.028     69.237      0.0

- Модель объясняет примерно 80% вариации продаж, что указывает на высокое качество.

- Большинство главных компонент статистически значимы (p < 0.05), то есть скрытые факторы, выделенные из исходных переменных действительно влияют на продажи. 

- первая- самый сильный положительный фактор, связанный с инновациями и технологиями.

- Некоторые (x2, x8) имеют отрицательное влияние, вероятно отражая издержки или конкуренцию.

Итого: инновации и технологическое развитие повышают продажи, а внешние и затратные факторы снижают.

Таблицы результатов PCA

In [12]:
explained_table = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(len(explained_ratio))],
    'Explained_Var_Ratio': explained_ratio,
    'Cumulative_Ratio': cum_ratio
})
display(explained_table.head(10))
display(explained_table.iloc[:m])

loadings = pd.DataFrame(PCA_coef, index=feature_names, columns=[f'PC{i+1}' for i in range(m)])
top_loadings = {}
for i in range(m):
    col = f'PC{i+1}'
    top_loadings[col] = loadings[col].abs().sort_values(ascending=False).head(5).index.tolist()
top_loadings, loadings


,PC,Explained_Var_Ratio,Cumulative_Ratio
0,PC1,0.287235,0.287235
1,PC2,0.119745,0.406980
2,PC3,0.071624,0.478604
3,PC4,0.066415,0.545019
4,PC5,0.063820,0.608839
5,PC6,0.063210,0.672050
6,PC7,0.062354,0.734404
7,PC8,0.055593,0.789997
8,PC9,0.049254,0.839251
9,PC10,0.044225,0.883476


,PC,Explained_Var_Ratio,Cumulative_Ratio
0,PC1,0.287235,0.287235
1,PC2,0.119745,0.406980
2,PC3,0.071624,0.478604
3,PC4,0.066415,0.545019
4,PC5,0.063820,0.608839
5,PC6,0.063210,0.672050
6,PC7,0.062354,0.734404
7,PC8,0.055593,0.789997
8,PC9,0.049254,0.839251
9,PC10,0.044225,0.883476


({'PC1': ['growth_innovation',
   'growth_technological',
   'lab',
   'share_RD',
   'joint'],
  'PC2': ['support',
   'fin_support',
   'share_equipments',
   'share_technology',
   'lab'],
  'PC3': ['climate', 'compet', 'joint', 'growth_marketing', 'patents'],
  'PC4': ['patents', 'growth_marketing', 'cooperation', 'age', 'compet'],
  'PC5': ['cooperation',
   'emp',
   'growth_marketing',
   'share_technology',
   'compet'],
  'PC6': ['emp', 'growth_marketing', 'patents', 'cooperation', 'age'],
  'PC7': ['age', 'share_technology', 'emp', 'patents', 'cooperation'],
  'PC8': ['patents',
   'growth_marketing',
   'cooperation',
   'emp',
   'share_technology'],
  'PC9': ['share_technology',
   'age',
   'cooperation',
   'share_equipments',
   'growth_innovation'],
  'PC10': ['share_technology',
   'share_equipments',
   'share_RD',
   'growth_technological',
   'growth_innovation']},
                            PC1       PC2       PC3       PC4       PC5  \
 growth_technological  0.3

- PC1: отражает инновационно-технологический потенциал (рост инноваций, технологий, НИОКР, лаборатории, совместные проекты).

- PC2: финансово-ресурсный фактор (поддержка, финансирование, оборудование, технологии).

- PC3–PC4 связаны с маркетингом, патентами и кооперацией, отражая активность на рынке.

- PC5–PC7: структура и опыт компании (возраст, занятость, технологии).

- PC8: комбинированный фактор маркетинг + кооперация.

Итого: основные движущие силы продаж — инновации, ресурсы и рыночное взаимодействие.

Прогнозы и базовые метрики на обучающей выборке

In [13]:
y_pred = model_pcr.predict(X_pca_const)
res_pcr = pd.DataFrame({'y_actual': y, 'y_pred': y_pred})
res_pcr.head()


,y_actual,y_pred
0,3.59,3.502626
1,1.28,0.818329
2,0.48,2.003198
3,0.86,1.847271
4,2.24,6.122497


Разброс между y_actual и y_pred умеренный. В целом- PCR-модель хорошо предсказывает продажи на основе скрытых факторов.

### 2. регрессии с L1 и L2 регуляризацией

In [14]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.metrics import r2_score, mean_squared_error

In [15]:
df = pd.read_excel('data/innovation_new.xlsx')
y = df['Sales'].values
X = df.drop(columns=['Sales']).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

alphas = np.logspace(-3, 3, 100)

lasso = LassoCV(alphas=alphas, cv=5, random_state=42).fit(X_scaled, y)
ridge = RidgeCV(alphas=alphas, cv=5).fit(X_scaled, y)

y_pred_lasso = lasso.predict(X_scaled)
y_pred_ridge = ridge.predict(X_scaled)

print("LASSO:")
print("Alpha:", lasso.alpha_)
print("R²:", r2_score(y, y_pred_lasso))
print("MSE:", mean_squared_error(y, y_pred_lasso))
print("Ненулевые коэффициенты:", np.sum(lasso.coef_ != 0))
print()

print("RIDGE:")
print("Alpha:", ridge.alpha_)
print("R²:", r2_score(y, y_pred_ridge))
print("MSE:", mean_squared_error(y, y_pred_ridge))


LASSO:
Alpha: 0.1
R²: 0.8099316233764201
MSE: 0.7450854596284087
Ненулевые коэффициенты: 8

RIDGE:
Alpha: 247.7076355991714
R²: 0.8131678980703476
MSE: 0.7323989661640898


- объясняется довольно большая доля вариации- болше 80%

- L1 выбрала 8 наиболее значимых признаков, отбросив остальные. Это говорит, что лишь часть факторов реально влияет на Sales. Модель проще и интерпретируемее.

- L2 использует все переменные, сглаживая их влияние. Она показывает чуть лучшее качество (R² = 0.813)

Итого- обе модели согласованно подтверждают сильную зависимость продаж от инновационных и ресурсных факторов.

### 3. Сравнение результатов моделей

- PCR: R² = 0.807
- LASSO: R² = 0.810 (8 значимых признаков)
- RIDGE: R² = 0.813

Все три модели объясняют примерно одинаковую долю вариации продаж (примерно 80%).

Вывод:
Лучшая модель — Ridge (L2-регуляризация),
так как она обеспечивает наивысшую точность и устойчивость при сохранении всех признаков и контроле мультиколлинеарности.